**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/schapiro-2024"], check=True)
    os.chdir("/content/encore/entries/schapiro-2024")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Schapiro et al. (2024) from Dryad dataset 1rn8pk129

**Paper.** Schapiro KA, Rittenberg JD, Kenngott M, Marder E (2024). Ih block reveals separation of
timescales in pyloric rhythm response to temperature changes in *Cancer borealis*. *eLife* 13:RP98844.
[doi:10.7554/elife.98844](https://doi.org/10.7554/elife.98844). The claims file was written from the
version of record (full text from Europe PMC, PMC11479588), with its Methods, figure legends,
Figure 1-figure supplement 1 and Table 1.

**Dataset.** [DRYAD:1rn8pk129](https://doi.org/10.5061/dryad.1rn8pk129), "Data from: Effects of Cs on
temperature response of C. borealis pyloric rhythm", version id 310115 (published 2024-08-09). The
paper cites the dataset's DOI, which always resolves to the newest version, and no version. Two
versions are published: 290160 (2024-04-23) and 310115. 310115 is the newest, and it is the one
that matches the paper as published: its README and analysis script add the voltage-clamp cells of
the revision (Figure 1-figure supplement 1 and the Ih numbers of the version of record), and only
VoltageClampData.mat, VoltageClampTraceData.mat (new), MakePaperFigures_Clean.m and README.md differ
between the two (same SHA-256 for every other file, from the public file lists). The files are
staged from Dryad, checked against their SHA-256, and read in place (docs/archives.md, "Dryad");
the digest of each file read is printed below.

**Code.** The dataset holds the paper's MATLAB code (MakePaperFigures_Clean.m,
CalculateFrequencyPhaseEtc.m, extractBurstFrequency_Final.m, Phase_Calc.m, ProbabilityUpDown.m). The
helper `pyloric.py` ports the parts used here; its docstring says where each parameter comes from.

**Claims.**

| Claim | Tested |
|---|---|
| figs1-ih-reduced-11c, figs1-ih-reduced-21c | blocked (data missing): VoltageClampData.mat holds one trial per experiment, no Cs+ trial |
| fig1-cs-lower-frequency-11c, fig1d-difference-grows-with-temperature, fig1e-q10-lower-in-cs | tested |
| fig1f-two-saline-frequency-change-same | tested on the deposited per-experiment table (spike data of 5 of 6 control experiments are not deposited) |
| fig1e-two-saline-q10-same, fig1e-saline-q10-same-across-designs | blocked (data missing): the criterion's Q10 reading cannot be computed without the control experiments' spike data |
| fig2c-decrease-by-depolarizing-phase | tested on the experiments with intracellular PD traces and spike times; the Cs+ + PTX example on its raw trace |
| fig3d, fig3e, fig4c, fig4d, fig4e | tested |
| fig5b, table1 (four claims), fig5d, lp-duty-cycle-preserved, matched-period-lp-earlier | tested |
| fig6a (three claims) | tested |

In [ ]:
import time
import warnings
import resource

import numpy as np
import pandas as pd
from scipy import stats

import pyloric as P

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
np.set_printoptions(precision=3, suppress=True, linewidth=200)
started = time.time()

S = P.load_mat("SummaryStatistics.mat")
EXPTS = ["_".join(str(np.ravel(x)[0]).split("_")[1:3]) for x in np.ravel(S["files"]["name"])]
PTX_EXPTS = ["969_147", "969_150", "969_152", "969_154", "981_035"]
spike_files = sorted(p.name for p in P.DATA_DIR.glob("*_SpikeAndTemp.mat"))
A = {f.split("_Spike")[0]: P.analyse(f.split("_Spike")[0]) for f in spike_files}
A["981_035_PD2"] = P.analyse("981_035", pd_key="PD2")
print(f"{len(spike_files)} SpikeAndTemp files analysed in {time.time() - started:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
print("The 21 saline/Cs+ experiments of SummaryStatistics.mat:", EXPTS)

In [ ]:
# SHA-256 of every file this notebook reads (the public file list of version 310115 gives the same).
read = spike_files + ["SummaryStatistics.mat", "SummaryStatisticsPTX.mat", "Sawtooth.mat", "SawtoothPTX.mat",
                      "LPAdvanceSamePeriod.mat", "VoltageClampData.mat", "VoltageClampTraceData.mat",
                      "MakePaperFigures_Clean.m"] + [f"Raw_Expt_{e}.mat" for e in ("969_087", "969_113", "969_115", "969_137", "969_147", "981_007", "981_037")]
for f in read:
    print(f"{P.sha256(P.DATA_DIR / f)}  {f}")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
rows = []
for e, d in A.items():
    c = d["cycles"]
    rows.append({"expt": e, "in_21": e in EXPTS, "PTX": e.split("_PD2")[0] in PTX_EXPTS,
                 "PD": len(d["PD2" if e.endswith("PD2") else "PD"]), "LP": len(d.get("LP", [])), "PY": len(d.get("PY", [])),
                 "spikes_end_min": round(d["PD"][-1] / 60, 1), "files": len(d["fileTimes"]),
                 "fileTimes_end_min": round(d["fileTimes"][-1] / 60, 1), "x_sub_end_min": round(d["x_sub"].max(), 1),
                 "Tempsub_dt_s": round(np.median(np.diff(d["x_sub"][:1000])) * 60, 3),
                 "Tempsub_min": round(np.nanmin(d["Tempsub"]), 1), "Tempsub_max": round(np.nanmax(d["Tempsub"]), 1),
                 "n_below_5C": int(np.sum(d["Tempsub"] < 5)), "cycles": len(c["freq"]),
                 "conditions": "; ".join(f"{k} {v[0].tolist()}-{v[1].tolist()}" for k, v in d["Timings"].items())})
info = pd.DataFrame(rows)
print(info.to_string(index=False))
print("\nExperiments with LP spikes:", int((info[info.in_21].LP > 0).sum()), "of 21; with PY spikes:",
      int((info[info.in_21].PY > 0).sum()), "of 21")

The README says spike times, a subsampled temperature (`Tempsub`) and its time (`x_sub`), the
condition table `Timings` (file indices) and the file start times (`fileTimes`). The values agree:
spike times are in seconds (they end at the last file time), `x_sub` is in minutes, sampled every
0.02 min (1.2 s), and file times are 120 s apart, as the Methods' 2 min files imply. The 21
experiments of the summary files are all deposited as spike files; each has PD and LP spikes, and
PY spikes are present in the number printed (the paper's Table 1 uses 12 with PY). Three files go
beyond this: 981_025 is a control experiment (two saline protocols, then Cs+ ramps), 981_035 and
969_154 are picrotoxin experiments outside the 21. 981_035 holds two PD spike trains, `PD` and
`PD2`, which the README does not mention. In two files (969_152, 981_010) the temperature runs on
after the sorted spikes end (to 278 and 142 min against 122 and 81 min); the spikes and the
temperature agree on the windows used (the step protocol in each window, below), so this only
means the temperature recording continued. Temperatures below 5 C (to -46.7 C), impossible in this
preparation, occur in three files: thermocouple dropouts, shown in the cell
`finding-temperature-dropouts`.

In [ ]:
# Temperature samples far outside the 10-26 C protocol, and whether they fall in an analysis window.
for e in ["969_070", "969_087", "969_089"]:
    d = P.load_spikes(e)
    t, x = d["Tempsub"], d["x_sub"][: len(d["Tempsub"])] * 60
    bad = np.nonzero(t < 5)[0]
    wins = [(d["fileTimes"][a[0]], d["fileTimes"][min(b[0] + 1, len(d["fileTimes"]) - 1)]) for a, b in d["Timings"].values()]
    inwin = sum(any(w0 < x[i] < w1 for w0, w1 in wins) for i in bad)
    print(f"{e}: {len(bad)} samples below 5 C, min {t.min():.1f} C, at {x[bad].min() / 60:.1f}-{x[bad].max() / 60:.1f} min; "
          f"{inwin} inside a condition window")

In [ ]:
# Is any recording deposited twice? Compare every pair of PD spike trains: fraction of spikes of one
# within 1 ms of a spike of the other (independent recordings give the chance level, a few percent).
def coincidence(a, b, tol=0.001):
    i = np.clip(np.searchsorted(a, b), 1, len(a) - 1)
    return float(np.mean(np.minimum(np.abs(a[i] - b), np.abs(a[i - 1] - b)) < tol))

names = sorted(A)
pd_trains = {e: A[e]["PD2" if e.endswith("PD2") else "PD"] for e in names}
co = pd.DataFrame([[coincidence(pd_trains[a], pd_trains[b]) if a != b else np.nan for b in names] for a in names], index=names, columns=names)
print("Largest PD coincidence of each train with any other file (1 ms):")
print(co.max(axis=1).round(3).sort_values(ascending=False).head(6).to_string())
lens = info.set_index("expt")[["PD", "LP", "PY", "cycles"]]
print("\nPairs with identical PD spike counts:", [(a, b) for i, a in enumerate(names) for b in names[i + 1:] if lens.PD[a] == lens.PD[b]])

No two files share their PD spike trains beyond chance, so no recording is deposited twice as a
spike file. 981_035's `PD` and `PD2` trains are not the same spikes either (the coincidence is
printed above); the next cell shows what they are. The picrotoxin summaries enter them as two
experiments: SummaryStatisticsPTX.mat as `BurstAvgs_981_035` and `BurstAvgs_981_035_2`, and
SawtoothPTX.mat with six columns for five files (both printed in the next cell).

In [ ]:
# 981_035 holds two PD spike trains, PD and PD2, and one temperature trace. Two PD neurons of one
# preparation are electrically coupled and burst together; two preparations burst independently.
SP = P.load_mat("SummaryStatisticsPTX.mat")
SWP = P.load_mat("SawtoothPTX.mat")
print("SummaryStatisticsPTX files:", [str(np.ravel(x)[0]) for x in np.ravel(SP["files"]["name"])])
print("SawtoothPTX files:", [str(np.ravel(x)[0]) for x in np.ravel(SWP["files"]["name"])], "DecFreq shape", np.asarray(SWP["DecFreq"]).shape)
print("Measured mean temperatures of the saline holds, rows 5 and 6 (981_035 and 981_035_2):")
print(np.asarray(SP["TotalActTemp_S"])[4:6].round(3))
d, d2 = A["981_035"], A["981_035_PD2"]
print("981_035 variables:", [k for k in ("PD", "PD2", "LP", "PY") if k in d and len(d[k])], "PD spikes", len(d["PD"]), "PD2 spikes", len(d["PD2"]))
s1, s2 = d["all_starts"], d2["all_starts"]
def nearest(a, b):
    i = np.clip(np.searchsorted(a, b), 1, len(a) - 1)
    return np.minimum(np.abs(a[i] - b), np.abs(a[i - 1] - b))
half = np.median(np.diff(s1)) / 2
for tol in (0.02, 0.05, 0.1):
    print(f"PD2 burst starts within {1e3 * tol:.0f} ms of a PD burst start: {np.mean(nearest(s1, s2) < tol):.3f}; "
          f"with PD2 shifted by half a median PD period ({half:.2f} s, chance): {np.mean(nearest(s1, s2 + half) < tol):.3f}")
print(f"PD and PD2 spikes within 1 ms of each other: {coincidence(d['PD'], d['PD2']):.3f} of PD2 spikes")
for cnd, (a, b) in d["Timings"].items():
    m1 = P.condition_mask(s1, d["fileTimes"], a[0], b[0]); m2 = P.condition_mask(s2, d["fileTimes"], a[0], b[0])
    print(f"{cnd}: bursts PD {m1.sum()}, PD2 {m2.sum()}; median burst frequency PD {1 / np.median(np.diff(s1[m1])):.2f} Hz, PD2 {1 / np.median(np.diff(s2[m2])):.2f} Hz")


The two trains do not burst together: PD2 bursts start near a PD burst start no more often than
when PD2 is shifted by half a period, and the two run at different frequencies in every condition
(printed above). Two PD neurons of one preparation would burst together, so `PD` and `PD2` are two
rhythms: two preparations recorded with one thermocouple, which is why the summaries give them the
same measured temperatures. The paper's six picrotoxin experiments are six preparations, as it
says. The README does not describe `PD2`, and nothing in the file says that it is a second
preparation.

In [ ]:
# Clock check. Spike times and temperature share a clock if pyloric frequency follows the temperature
# steps with no delay: correlate each cycle's frequency with the temperature k minutes later
# (Tempsub interpolated at the cycle start + k), in the saline window of each experiment. With 4 min
# holds and 2 C steps, a clock offset of a minute or more moves the peak away from 0.
shifts = np.arange(-6, 6.01, 0.25)
res = []
for e in EXPTS:
    d = A[e]
    c = d["cycles"]
    a, b = d["Timings"]["Saline"]
    m = np.nonzero(P.condition_mask(c["start"], d["fileTimes"], a[0], b[0]) & (c["freq"] > 1 / 8))[0]
    tt, TT = d["x_sub"][: len(d["Tempsub"])] * 60, d["Tempsub"]
    ok = ~np.isnan(TT) & (TT > 5)
    r = [np.corrcoef(c["freq"][m], np.interp(c["start"][m] + k * 60, tt[ok], TT[ok]))[0, 1] for k in shifts]
    res.append({"expt": e, "best_shift_min": shifts[int(np.nanargmax(r))], "r_at_0": r[int(np.argmin(np.abs(shifts)))], "r_best": np.nanmax(r),
                "r_at_-3min": r[int(np.argmin(np.abs(shifts + 3)))], "r_at_+3min": r[int(np.argmin(np.abs(shifts - 3)))]})
res = pd.DataFrame(res)
print(res.round(3).to_string(index=False))
print("\nmedian best shift:", res.best_shift_min.median(), "min; experiments with |best shift| <= 0.5 min:",
      int((res.best_shift_min.abs() <= 0.5).sum()), "of", len(res), "; correlation lower at +3 min than at 0 in",
      int((res["r_at_+3min"] < res.r_at_0).sum()), "of", len(res))

The correlation of each cycle's frequency with the temperature at the same time is high in most
experiments (r at 0 printed above) and drops when the temperature is taken 3 min later in most of
them (count printed above). The best shift itself is within 0.5 min of 0 in only some experiments
and spreads over several minutes, because the frequency follows the temperature with a delay and
stays near a plateau over the 4 min holds, so the slow protocol cannot place it more precisely. This
check is weak: it is consistent with a shared clock, not proof of it. The support for a shared clock
is the raw traces of 969_113 (next cell) and the close agreement of the per-experiment values
recomputed here from spikes and temperature with the deposited SummaryStatistics.mat (below), which
needs spikes and temperature on one clock.

In [ ]:
# Raw_Expt_969_113: intracellular PD, sample 0 at t = 0 of the spike times? The mean PD voltage
# around the PD spike times should peak at lag 0.
d = A["969_113"]
t0, t1 = 3800.0, 3860.0
v, n_samples = P.raw_trace("969_113", "PD", t0, t1)
sp = d["PD"][(d["PD"] > t0 + 0.05) & (d["PD"] < t1 - 0.05)]
shifts = np.arange(-20, 21)  # in 0.1 ms samples
prof = [np.mean(v[np.round((sp - t0) * 1e4).astype(int) + s]) for s in shifts]
print(f"Raw_Expt_969_113: {n_samples} samples = {n_samples / 1e4 / 60:.1f} min at 10 kHz; x_sub of 969_113 ends at {d['x_sub'].max():.1f} min")
print(f"{len(sp)} PD spikes in {t0:.0f}-{t1:.0f} s; mean PD voltage at the spike time {prof[20]:.1f} mV, "
      f"peak at shift {shifts[int(np.argmax(prof))] / 10:.1f} ms; voltage 20 ms before the spikes {np.mean(v[np.round((sp - t0) * 1e4).astype(int) - 200]):.1f} mV")
tr, _ = P.raw_trace("969_113", "temp", t0, t1)
c = d["cycles"]
k = (c["start"] > t0) & (c["start"] < t1)
print(f"temperature channel {tr.mean():.2f} C over the window; Tempsub cycle mean {np.nanmean(c['temp'][k]):.2f} C")

The intracellular PD voltage peaks within a fraction of a millisecond of the deposited PD spike
times, and the raw temperature channel equals `Tempsub` over the same window: the raw trace, the
spike times and the temperature share one clock in 969_113.

In [ ]:
# Agreement of this port of the deposited code with the deposited per-experiment values, with the
# cycles left by the outlier removal before a removed cycle (extract_bursts, `gap`) kept, as the
# deposited code keeps them, and left out, the main reading here.
def dep(name, rows=slice(None), cols=slice(2, 8)):
    return np.asarray(S[name])[rows, cols]

CS = {e: ("Cs" if "Cs" in A[e]["Timings"] else "Cs_1") for e in EXPTS}
ng = np.array([A[e]["cycles"]["gap"].sum() for e in EXPTS]); nc = np.array([len(A[e]["cycles"]["gap"]) for e in EXPTS])
print(f"gap cycles: {ng.sum()} of {nc.sum()} cycles ({100 * ng.sum() / nc.sum():.1f}%), {ng.min()}-{ng.max()} per experiment")
c = A["969_128"]["cycles"]; g = np.nonzero(c["gap"])[0]
j = g[np.nanargmax(c["LP_ON"][g])]
print(f"example, 969_128 cycle {j}: period kept {c['period'][j]:.2f} s, next start kept {A['969_128']['all_starts'][j + 1] - c['start'][j]:.2f} s later, LP ON phase {c['LP_ON'][j]:.2f}")
spec = [("TotalSals", "freq", "S"), ("TotalLPON_S", "LP_ON", "S"), ("TotalPDOFF_S", "PD_OFF", "S"), ("TotalLPOFF_S", "LP_OFF", "S"),
        ("TotalExpts", "freq", "C"), ("TotalLPON_E", "LP_ON", "C"), ("TotalPDOFF_E", "PD_OFF", "C"), ("TotalLPOFF_E", "LP_OFF", "C")]
big = []
for dg in (False, True):
    print(f"\ngap cycles {'left out (main reading)' if dg else 'kept (deposited code)'}:")
    for k, m, cnd in spec:
        hs = [P.hold_means(A[e], "Saline" if cnd == "S" else CS[e], strict_cs=(cnd == "C" and m == "freq"), drop_gaps=dg) for e in EXPTS]
        v = np.array([h[m] for h in hs])
        dd = np.abs(v - dep(k))
        print(f"  {k:14s} values {np.sum(~np.isnan(dd)):3d}; |this - deposited| median {np.nanmedian(dd):.4f}, 90th percentile {np.nanpercentile(dd, 90):.4f}, max {np.nanmax(dd):.3f}; "
              f"NaN here only {int(np.sum(np.isnan(v) & ~np.isnan(dep(k))))}, deposited only {int(np.sum(~np.isnan(v) & np.isnan(dep(k))))}")
        if dg:
            for i, jj in zip(*np.nonzero(dd > 0.04)):
                f_ = A[EXPTS[i]]["cycles"]["freq"][hs[i]["cycle_index"][int(P.TARGETS[jj])]]
                big.append({"value": k, "expt": EXPTS[i], "target_C": int(P.TARGETS[jj]), "here": v[i, jj], "deposited": dep(k)[i, jj], "cycles": hs[i]["n"][jj],
                            "cycle freq CV": np.std(f_) / np.mean(f_)})
print("\nDifferences above 0.04 that remain with the gap cycles left out, with the cycle-to-cycle CV of frequency in those cycles:")
print(pd.DataFrame(big).round(3).to_string(index=False))
allcv = [np.std(A[e]["cycles"]["freq"][h]) / np.mean(A[e]["cycles"]["freq"][h]) for e in EXPTS for cnd in ("Saline", CS[e])
         for h in P.hold_means(A[e], cnd)["cycle_index"].values() if len(h) > 1]
print(f"cycle-to-cycle CV of frequency over all holds: median {np.median(allcv):.3f}, 90th percentile {np.percentile(allcv, 90):.3f}")


Recomputed from the spike files with the deposited code's rules, the per-experiment frequencies and
phases agree with the deposited SummaryStatistics.mat to within a few thousandths in most values,
with a few larger differences (printed above). The outlier removal of the deposited
extractBurstFrequency_Final.m deletes a cycle's start together with its frequency, so the cycle
before it keeps its own period while its phases are measured up to a start two or more bursts later
(in the 969_128 example, an LP ON phase of 32). These cycles are 1.6% of all cycles. With them left
out, the median phase difference from the deposited values is five to twelve times smaller and the
largest phase difference falls from 0.85 to 0.11, while the frequencies barely change; so the
summary phases were most likely made by code that did not include these cycles. This attempt leaves
them out of the hold means and phases (an assumption; each claim's section shows the deposited
code's reading, gaps kept, as a variant). Of the 12 differences above 0.04 that remain, 10 are in
holds where the rhythm is irregular from cycle to cycle (frequency CV 0.25 to 0.57, against a median
of 0.037 over all holds), so a small change in which cycles enter moves the mean; the other two,
969_147 at 21 C in saline (CV 0.023) and in Cs+ (CV 0.11), are not explained. All claims below are computed from
the spike files.

In [ ]:
# The outlier removal of the deposited extractBurstFrequency_Final.m, printed for comparison with the
# port (pyloric.extract_bursts).
src_m = (P.DATA_DIR / "extractBurstFrequency_Final.m").read_text().splitlines()
i0 = next(i for i, l in enumerate(src_m) if l.startswith("%Some code"))
print("\n".join(f"{i + 1:3d}  {l}" for i, l in enumerate(src_m) if i >= i0 and l.strip()))

These are the lines that `extract_bursts` ports: the same loop over P = 11 to length(burstFreq) - 10,
the same windows (P-10 to P-5 and P+5 to P+10), the same threefold criterion on both sides, the
removal repeated until no cycle is flagged, and StartTime(P) deleted together with burstFreq(P),
cyclePeriod(P) and burstDuration(P). Deleting StartTime(P) leaves cycle P-1 with its own period
while the next start kept is that of burst P+1: these are the gap cycles of the cell above. The
deposited code thus makes them; the summaries' phases agree with leaving them out (above).

In [ ]:
# The two-saline control experiments (Figure 1E, F): which are deposited?
CT = S["ControlTable"]
print(CT.to_string())
ctl = [str(np.ravel(x)[0]) for x in CT["Expt"] if np.size(x)]
print("\nControl experiments:", ctl, "; with a SpikeAndTemp file:", [e for e in ctl if f"{e}_SpikeAndTemp.mat" in spike_files])
print("Second protocol type:", dict(zip(ctl, [str(np.ravel(x)[0]) for x in CT["Type_2"] if np.size(x)])))

ControlTable lists six control experiments; only 981_025 has a spike file. The other five
(965_129, 965_134, 965_139, 965_146, 982_005) are present only as the table's per-experiment means
(11 and 21 C frequencies, Q10, frequency change). Rows 7 and 8 of the table hold no experiment
(their values equal the column means and SDs of rows 1 to 6). The table says the second protocol
was a ramp, not steps, in three of the six (965_146, 981_025, 982_005); the paper says "two
consecutive temperature changes in saline".

In [ ]:
# Voltage clamp: what VoltageClampData.mat holds against what its ConditionsKey describes.
VC = P.load_mat("VoltageClampData.mat")["VoltageClampData"]
for n in VC.dtype.names:
    e = VC[n].ravel()[0]
    key = np.ravel(e["ConditionsKey"].ravel()[0]).astype(int)
    desc = [str(np.ravel(c)[0]) for c in np.ravel(e["ConditionDescriptions"].ravel()[0])]
    V = e["Voltage"].ravel()[0]
    ncs = sum("cs" in desc[k - 1].lower() for k in key)
    print(f"{n}: ConditionsKey lists {len(key)} trials, {ncs} of them Cs+; Voltage array {V.shape} (trials x steps x samples); "
          f"trial 1 is condition {key[0]} '{desc[key[0] - 1]}'")
VT = P.load_mat("VoltageClampTraceData.mat")["VoltageClampDataTraces"]
print("\nVoltageClampTraceData:", {n: [(str(np.ravel(r['Cell_Temperature'])[0]), str(np.ravel(r['Condition'])[0]), str(np.ravel(r['fileNames'])[0])) for r in VT[n].ravel()[0].ravel()] for n in VT.dtype.names})

VoltageClampData.mat holds one trial per experiment, the first, whose condition is saline or
TTX+PTX in every experiment, while its ConditionsKey lists 5 to 22 trials, 2 to 8 of them Cs+.
The README says "Trials x steps x time". No Cs+ voltage-clamp trial is in the file. The Cs+ traces of
the dataset are the four cells of VoltageClampTraceData.mat (two at 11 C, two at 21 C), the example
cells of Figure 1-figure supplement 1. In version 290160 the file was 814 MB, against 98 MB in
310115; it may have held every trial, which this attempt cannot read (it is not staged).

## figs1-ih-reduced-11c and figs1-ih-reduced-21c

> At 11 degrees C and -110 mV, Ih in PD and LP cells was 1.5 nA in saline and 0.23 nA in Cs+, a
> reduction of 93% (4 PD and 3 LP cells, paired t-test p = 0.0005).
>
> At 21 degrees C and -110 mV, Ih was 3.2 nA in saline and 0.6 nA in Cs+, a reduction of 78% (5 LP
> and 3 PD cells, paired t-test p = 0.02).

Blocked: the Cs+ trials of 7 cells at 11 C and 8 at 21 C are not in the dataset (above). The two
criterion readings are run on the four cells that are deposited with both conditions; and the
deposited script's own per-cell values, typed in MakePaperFigures_Clean.m from by-eye readings, are
parsed from the script to show what the paper's numbers rest on. Neither decides the outcome.

In [ ]:
def ih_readings(t, i, step_on, step_len=12.0):
    """Ih at a hyperpolarising step: initial minus steady-state current (criterion; the deposited
    script's Leak - Sag). The capacitive transient is taken to end 20 ms after the step (standard
    choice); steady state the mean of the last 1 s of the step. Initial current: (a) the mean of the
    first 100 ms after the transient; (b) the current farthest from the steady state in the first
    500 ms, after a 10 ms moving average (standard choice against noise); (c) the least inward
    current (10 ms moving average) in the first 1 s, the point before the slow inward Ih develops,
    after a faster current has relaxed (printed below; a standard choice, closest to the script's
    "initial current" read by eye)."""
    fs = 1 / np.median(np.diff(t))
    a = int(round((step_on + 0.02) * fs))
    ss = np.mean(i[int((step_on + step_len - 1.0) * fs): int((step_on + step_len) * fs)])
    ia = np.mean(i[a: a + int(0.1 * fs)])
    k = int(0.01 * fs)
    w = np.convolve(i[a: a + int(0.5 * fs)], np.ones(k) / k, mode="valid")
    ib = w[int(np.argmax(np.abs(w - ss)))]
    ic = np.max(np.convolve(i[a: a + int(1.0 * fs)], np.ones(k) / k, mode="valid"))
    return ia - ss, ib - ss, ic - ss

rows = []
for n in VT.dtype.names:
    for r in VT[n].ravel()[0].ravel():
        t = np.ravel(r["Time"])
        V = np.asarray(r["Voltage"])[:, 0, :]
        I = np.asarray(r["Current"])[:, 0, :]
        lvl = np.round(np.median(V[int(12 * 1e4): int(18 * 1e4)], axis=0))
        k = int(np.argmin(np.abs(lvl + 110)))
        below = np.nonzero(V[:, k] < (V[: int(1e4), k].mean() + lvl[k]) / 2)[0]
        onset, offset = t[below[0]], t[below[-1]]
        a, b, c_ = ih_readings(t, I[:, k], step_on=onset)
        fs = 1e4
        early = [np.mean(I[int((onset + x) * fs): int((onset + x + 0.05) * fs), k]) for x in (0.02, 0.1, 0.3, 0.6, 1.0, 4.0, 8.0, 11.0)]
        rows.append({"expt": n, "cell": str(np.ravel(r["Cell_Temperature"])[0]), "cond": str(np.ravel(r["Condition"])[0]),
                     "step_onset_s": round(onset, 3), "step_len_s": round(offset - onset, 3), "step_mV": lvl[k], "Ih_a_nA": a, "Ih_b_nA": b, "Ih_c_nA": c_,
                     "I at +0.02, 0.1, 0.3, 0.6, 1, 4, 8, 11 s (nA)": " ".join(f"{x:.2f}" for x in early)})
        if n == "Expt_981_064" and rows[-1]["cond"] == "Saline":
            print(f"{n} LP saline, step to {lvl[k]:.0f} mV: V and I in 0.5 s bins over the whole trace (0-{t[-1]:.0f} s)")
            print("  V (mV):", np.round([V[j: j + 5000, k].mean() for j in range(0, len(t), 5000)], 0).tolist())
            print("  I (nA):", np.round([I[j: j + 5000, k].mean() for j in range(0, len(t), 5000)], 2).tolist())
ih = pd.DataFrame(rows)
print(ih.round(3).to_string(index=False))
w = ih.pivot_table(index=["expt", "cell"], columns="cond", values=["Ih_a_nA", "Ih_b_nA", "Ih_c_nA"])
for r_ in ("a", "b", "c"):
    red = 100 * (1 - w[(f"Ih_{r_}_nA", "Cs")] / w[(f"Ih_{r_}_nA", "Saline")])
    print(f"reading ({r_}) reduction per cell (%):", red.round(1).to_dict())


In [ ]:
# The deposited script's per-cell values (MakePaperFigures_Clean.m, "Stats on Change in Ih"):
# Ih = mean over repeats of (Leak - Sag) at -100, -110, -120 mV, typed from by-eye readings.
import re
src = (P.DATA_DIR / "MakePaperFigures_Clean.m").read_text()
def mat_array(name):
    m = re.search(rf"^{name}=\[(.*?)\]", src, re.S | re.M)
    body = re.sub(r"%[^\n]*", "", m.group(1)).replace("...", "")
    return np.array([[float(x) if x != "nan" else np.nan for x in re.split(r"[ ,]+", r.strip()) if x] for r in body.split(";") if r.strip()])
def cell_ih(prefix):
    return np.nanmean(mat_array(prefix.replace("#", "Leak")) - mat_array(prefix.replace("#", "Sag")), axis=0)
for T, cells_ in ((11, ["PD_149", "LP_152", "PD_156", "PD_027", "LP_030", "LP_064", "PD_065"]),
                  (21, ["PD_149", "LP_152", "LP_027", "LP_033", "PD_033", "LP_092", "LP_064", "PD_093"])):
    sal = np.array([cell_ih(f"{c}_#_{T}_Sal") for c in cells_])[:, 1]
    cs = np.array([cell_ih(f"{c}_#_{T}_Cs") for c in cells_])[:, 1]
    red = 100 * (sal - cs) / sal
    print(f"{T} C, -110 mV, {len(cells_)} cells (script values): saline {np.mean(sal):.2f} +- {np.std(sal, ddof=1):.2f} nA, "
          f"Cs {np.mean(cs):.2f} +- {np.std(cs, ddof=1):.2f} nA, reduction {np.mean(red):.0f} +- {np.std(red, ddof=1):.0f} %, "
          f"paired t p = {stats.ttest_rel(sal, cs).pvalue:.4f}")
# The four cells with deposited traces, against the script's by-eye values at -110 mV.
ihx = ih.set_index(["expt", "cond"])
for e, c_, T in (("Expt_981_065", "PD_065", 11), ("Expt_981_064", "LP_064", 11), ("Expt_970_093", "PD_093", 21), ("Expt_981_033", "LP_033", 21)):
    print(f"{e} ({c_}, {T} C): script saline {cell_ih(f'{c_}_#_{T}_Sal')[1]:.2f}, Cs {cell_ih(f'{c_}_#_{T}_Cs')[1]:.2f} nA; "
          + "; ".join(f"here ({r_}) saline {ihx.loc[(e, 'Saline'), f'Ih_{r_}_nA']:.2f}, Cs {ihx.loc[(e, 'Cs'), f'Ih_{r_}_nA']:.2f}" for r_ in "abc"))


**Result: blocked (data missing).** Ih cannot be measured on the paper's cells: the dataset holds
the Cs+ traces of four cells only, two per temperature. The steps are placed correctly (each lasts
12.000 s at -109 or -110 mV, printed above). Readings (a) and (b) give negative values, because the
current in the first few hundred ms after the step is a faster inward current that relaxes (in the
LP cell at 11 C, -22.9 nA at 20 ms and -20.4 nA at 0.6 s, before the slow Ih develops to -21.4 nA)
and in Cs+ the current goes on relaxing over the step. Reading (c), the least inward current in the
first second, gives saline Ih of 0.98 to 2.22 nA and Cs+ Ih of -0.07 to 0.05 nA in the four cells
(reductions of 98 to 107%), close to the script's by-eye values for the same cells (saline 0.50 to
2.18 nA, Cs+ -0.30 to 0.14 nA). Two cells per temperature cannot test the claim. The script's typed
per-cell values give 1.55 and 0.25 nA, 94% and p = 0.0005 at 11 C, and 3.17 and 0.58 nA, 78% and
p = 0.021 at 21 C, the paper's numbers; they are readings by eye of traces that are not deposited,
so they are shown, not tested.

## fig1-cs-lower-frequency-11c

> At 11 degrees C pyloric frequency was lower in Cs+ than in saline (0.9 against 1.2 Hz, paired
> t-test p < 0.001).

Per experiment and condition, the mean PD burst frequency over cycles within 0.3 C of 11 C in the
first 11 C hold of the condition (O2). The condition windows of the Timings table end at the 21 C
hold, so the return to 11 C (the O2 variant) is outside every window, during the solution change,
and cannot be used; the deposited code's reading, every 11 C cycle of the window, is the variant.
The deposited CalculateFrequencyPhaseEtc.m takes the Cs+ frequency over |T - target| < 0.3 C and
the saline frequency over <= 0.3 C; the main reading follows it, and `first_le` uses <= for both.
`first_gaps` keeps the cycles left by the outlier removal (data check).

In [ ]:
# Readings: first 11 C hold (main), every 11 C cycle of the window (the deposited code's), the
# first hold with <= 0.3 C for Cs+ as for saline, and the first hold with the gap cycles kept.
HV = {"first": ("first", True, True), "all": ("all", True, True), "first_le": ("first", False, True), "first_gaps": ("first", True, False)}
H = {w: {e: (P.hold_means(A[e], "Saline", which=wh, drop_gaps=dg), P.hold_means(A[e], CS[e], which=wh, strict_cs=st_, drop_gaps=dg)) for e in EXPTS}
     for w, (wh, st_, dg) in HV.items()}
for w in HV:
    fs = np.array([H[w][e][0]["freq"][0] for e in EXPTS]); fc = np.array([H[w][e][1]["freq"][0] for e in EXPTS])
    ns = [H[w][e][0]["n"][0] for e in EXPTS]; nc = [H[w][e][1]["n"][0] for e in EXPTS]
    r = P.paired(fs, fc)
    print(f"[{w}] saline {r['mean_x']:.2f} +- {r['sd_x']:.2f} Hz, Cs {r['mean_y']:.2f} +- {r['sd_y']:.2f} Hz, n pairs {r['n']}, "
          f"paired t = {r['t']:.2f}, p = {r['p_t']:.2g}; Cs lower in {int(np.sum(fc < fs))}; cycles per experiment saline {min(ns)}-{max(ns)}, Cs {min(nc)}-{max(nc)}")
    print("   experiments without an 11 C Cs value:", [e for e, v in zip(EXPTS, fc) if np.isnan(v)])
f11 = pd.DataFrame({"saline": [H["first"][e][0]["freq"][0] for e in EXPTS], "Cs": [H["first"][e][1]["freq"][0] for e in EXPTS]}, index=EXPTS)
print(f11.round(3).T.to_string())

**Result: reproduced.** In the first 11 C hold the mean frequency is 1.17 +- 0.15 Hz in saline and
0.86 +- 0.22 Hz in Cs+ over the 20 experiments with both values, within the tolerances of 1.2 and
0.9 Hz; Cs+ is lower in all 20, paired t-test p = 8e-07. The deposited code's reading (every 11 C
cycle of the window) selects the same cycles, because every window ends at the 21 C hold; the <=
bound for Cs+ and the gap cycles kept give the same values. 969_070 has no Cs+ cycle within 0.3 C
of 11 C.

## fig1d-difference-grows-with-temperature

> The difference between a preparation's frequency in saline and in Cs+ increased significantly as
> temperature increased.

Saline minus Cs+ mean frequency at each holding temperature, one-way repeated-measures ANOVA on
temperature: (a) experiments with all six values; (b) the 21, dropping incomplete ones as MATLAB
ranova does. The two give the same set.

In [ ]:
for w in HV:
    D = np.array([H[w][e][0]["freq"] - H[w][e][1]["freq"] for e in EXPTS])
    F, d1, d2, p, _, n = P.rm_anova(D)
    print(f"[{w}] experiments complete {n} of 21 (incomplete: {[e for e, r in zip(EXPTS, D) if np.isnan(r).any()]}); "
          f"F({d1},{d2}) = {F:.2f}, p = {p:.2g}")
    print("   mean difference by temperature (Hz):", dict(zip(P.TARGETS.tolist(), np.nanmean(D, axis=0).round(3))))
Dd = np.asarray(S["TotalDiffs"])[:, 2:8]
F, d1, d2, p, _, n = P.rm_anova(Dd)
print(f"deposited TotalDiffs: F({d1},{d2}) = {F:.3f}, p = {p:.2g} (paper F(5,85) = 43.259)")

**Result: reproduced.** The mean saline minus Cs+ difference grows from 0.31 Hz at 11 C to 0.83 Hz
at 21 C, and the temperature effect is significant, F(5,90) = 44.1, p = 9e-23, over the 19
experiments with all six values (969_070 and 969_128 are incomplete); every variant gives F >= 44.0.
The deposited TotalDiffs gives the paper's F(5,85) = 43.259 from 18 complete experiments; here
969_068 has an 11 C Cs+ value that the deposited table lacks.

## fig1e-q10-lower-in-cs

> The Q10 of pyloric frequency was lower in Cs+ (1.3) than in saline (1.7), paired t-test p <
> 0.001, N=21.

Q10 from the six holding-temperature means under O1: (a) the fitted Q10 equation (main), (b) log-linear
fit (the deposited code), (c) two-point ratio.

In [ ]:
Q = {}
for w in HV:
    for r_ in "abc":
        qs = np.array([P.q10(H[w][e][0]["temp"], H[w][e][0]["freq"], r_) for e in EXPTS])
        qc = np.array([P.q10(H[w][e][1]["temp"], H[w][e][1]["freq"], r_) for e in EXPTS])
        Q[(w, r_)] = (qs, qc)
        t = P.paired(qs, qc)
        print(f"[{w}, {r_}] saline {t['mean_x']:.2f} +- {t['sd_x']:.2f}, Cs {t['mean_y']:.2f} +- {t['sd_y']:.2f}, n {t['n']}, "
              f"paired t p = {t['p_t']:.2g}, Wilcoxon p = {t['p_w']:.2g}; Cs lower in {int(np.sum(qc < qs))}")
print("deposited Q10Sal mean %.2f, Q10Expt mean %.2f" % (np.mean(S["Q10Sal"]), np.mean(S["Q10Expt"])))

**Result: reproduced.** Under reading (a), mean Q10 is 1.68 +- 0.32 in saline and 1.35 +- 0.45 in
Cs+ (21 experiments), within the tolerances of 1.7 and 1.3, paired t-test p = 4.1e-05, Cs+ lower in
18. Readings (b), the deposited code's (1.71 against 1.35, p = 0.00011), and (c) (1.71 against 1.34,
p = 0.00014, 19 experiments) agree, as do the variants of the 11 C hold.

## fig1f-two-saline-frequency-change-same

> In experiments with two consecutive saline step protocols, the frequency change from 11 to 21
> degrees C did not differ between the first (0.9 Hz) and second (0.8 Hz) protocol.

A claim of absence, paired t-test over the 6 experiments. Five of the six are deposited only as
ControlTable's per-experiment values, so the test uses the table's 21 C minus 11 C frequency
(`Deg_21 - Deg_11`, equal to `FreqDiff`). For 981_025, the one with spikes, the value is recomputed
from the spike file.

In [ ]:
ct = CT.iloc[:6]
d1 = ct.Deg_21_1.astype(float) - ct.Deg_11_1.astype(float)
d2 = ct.Deg_21_2.astype(float) - ct.Deg_11_2.astype(float)
print(pd.DataFrame({"expt": ctl, "change_1": d1.values, "FreqDiff_1": ct.FreqDiff_1.values, "change_2": d2.values, "FreqDiff_2": ct.FreqDiff_2.values}).round(3).to_string(index=False))
t = stats.ttest_rel(d1, d2)
print(f"first {d1.mean():.2f} +- {d1.std(ddof=1):.2f} Hz, second {d2.mean():.2f} +- {d2.std(ddof=1):.2f} Hz, mean difference {np.mean(d1 - d2):.3f} Hz, "
      f"paired t({len(d1) - 1}) = {t.statistic:.2f}, p = {t.pvalue:.3f}")
# 981_025 from its spike file: conditions 'Saline Steps' and 'Saline Ramp'
d = A["981_025"]
for cond in ("Saline Steps", "Saline Ramp"):
    h = P.hold_means(d, cond, which="first")
    print(f"981_025 {cond}: 11 C {h['freq'][0]:.3f} Hz (n {h['n'][0]}), 21 C {h['freq'][-1]:.3f} Hz (n {h['n'][-1]}), change {h['freq'][-1] - h['freq'][0]:.3f} Hz")

**Result: reproduced.** The changes are 0.89 +- 0.30 Hz (first) and 0.78 +- 0.30 Hz (second),
within the tolerances, and the paired t-test does not find a difference, t(5) = 1.77, p = 0.137
(mean difference 0.11 Hz). The test rests on the deposited per-experiment values: the spike data of
five of the six experiments are not deposited. For 981_025 the spike file gives a change of 1.22 Hz
for the first protocol and 1.00 Hz for the second, against the table's 1.19 and 0.99.

## fig1e-two-saline-q10-same and fig1e-saline-q10-same-across-designs

> In the two-saline experiments the Q10 of the first protocol (2.0) did not differ from that of the
> second (1.8).
>
> The saline Q10 did not differ between experiments with two saline protocols and those with one
> saline and one Cs+ protocol.

Blocked: both criteria judge on Q10 under reading (a), the fit to the six holding-temperature means,
which needs the control experiments' spike data; five of the six are not deposited. The deposited
Q10 values (made by the deposited code, reading (b)) and reading (c) from the table's 11 and 21 C
means are shown as variants.

In [ ]:
q1, q2 = ct.Q10_1.astype(float).values, ct.Q10_2.astype(float).values
c1, c2 = (ct.Deg_21_1 / ct.Deg_11_1).astype(float).values, (ct.Deg_21_2 / ct.Deg_11_2).astype(float).values
for lab, a, b in (("deposited Q10 (b)", q1, q2), ("reading (c) from table", c1, c2)):
    t = stats.ttest_rel(a, b)
    print(f"{lab}: first {a.mean():.2f} +- {a.std(ddof=1):.2f}, second {b.mean():.2f} +- {b.std(ddof=1):.2f}, paired t p = {t.pvalue:.3f}")
sal21_b = Q[("first", "b")][0]
for lab, x in (("mean of the two", (q1 + q2) / 2), ("first only", q1), ("12 pooled", np.r_[q1, q2])):
    t = stats.ttest_ind(x, sal21_b)
    print(f"across designs, deposited control Q10 {lab} (n {len(x)}, mean {x.mean():.2f}) vs 21 saline Q10 (b) here (mean {np.nanmean(sal21_b):.2f}): "
          f"two-sample t p = {t.pvalue:.3f}")
t = stats.ttest_ind((q1 + q2) / 2, np.ravel(S["Q10Sal"]))
print(f"deposited control mean of two vs deposited Q10Sal: p = {t.pvalue:.3f} (paper p = 0.26)")

**Result: blocked (data missing).** The criterion's Q10 cannot be computed for five of the six
control experiments. Under the variants that can be run, neither comparison finds a difference:
the deposited Q10 values give 1.98 against 1.83 (paired t p = 0.144) and reading (c) from the table
1.92 against 1.79 (p = 0.133); across designs, p = 0.244 against the 21 saline Q10 values computed
here and p = 0.259 against the deposited ones (the paper's p = 0.26).

## fig2c-decrease-by-depolarizing-phase

> In an example preparation in Cs+, the frequency decrease right after the temperature step from 13
> toward 15 degrees C came from an elongation of the PD depolarizing phase, not of the bursting,
> repolarization or inhibitory phases; a Cs+ + PTX example showed the same.

The criterion's windows: the cycles of the 30 s before the 13 to 15 C step against the cycles of
the first 30 s of warming whose period exceeds the pre-step mean. The step onset is the first cycle
of the warming (the first cycle above 13.3 C, traced back while the smoothed temperature rose by
more than 0.01 C per cycle; a standard choice built on the paper's increasing-temperature rule).
Reading (a), three phases on the intracellular PD trace (voltage minimum from a 5 ms moving average,
a standard choice); reading (b), four phases from PD and LP spikes. Intracellular PD traces with
spike times are deposited for two Cs+ experiments, 969_113 (the paper's Figure 2 example, plotted
by the deposited script at 67.9 min) and 969_115. Reading (b) needs only spikes and is also run on
the other experiments the summary files mark as intracellular, and on all 21 as a variant. The
statement's Cs+ + PTX example lies outside the criterion, which takes the Cs+ experiments; it is
run as a variant. The deposited MakePaperFigures_Clean.m plots it from Raw_Expt_981_037.mat
("Figure 5F,G: example of Jag in PTX"; the script's figure numbers run one ahead of the paper's, its
"Stats for Fig 5C,D,E" being the paper's Figure 4C to E), at 16.06 min for 90 s. That file holds an
intracellular PD trace and the temperature, and no spike file is deposited for 981_037, so PD spikes
are detected on the trace (`pyloric.trace_spikes`) and reading (a) is run on the warming at 16 min,
which goes from 11 C toward 14 C in that recording, not from 13 C. The seven raw files and their
channels are listed first. Reading (b) is also run on 969_152 in Cs+ + PTX.

In [ ]:
def phase_shares(e, cond, trace=False, t_from=13, t_to=15, win=30.0, keep_gaps=False):
    d = A[e]
    c = d["cycles"]
    s_idx, e_idx = d["Timings"][cond]
    m = P.condition_mask(c["start"], d["fileTimes"], s_idx[0], e_idx[0])
    k = P.step_onset(c, m, t_from, t_to)
    if k is None:
        return None
    t_on = c["start"][k]
    pre = np.nonzero(m & (c["start"] >= t_on - win) & (c["start"] < t_on))[0]
    post = np.nonzero(m & (c["start"] >= t_on) & (c["start"] < t_on + win))[0]
    if not keep_gaps:
        pre, post = pre[~c["gap"][pre]], post[~c["gap"][post]]
    post = post[c["period"][post] > np.mean(c["period"][pre])]
    ph = {"burst": c["pd_dur"], "repol_b": c["LP_lat_on"] - c["pd_dur"], "inhib_b": c["LP_lat_off"] - c["LP_lat_on"],
          "depol_b": c["period"] - c["LP_lat_off"]}
    if trace:
        v, _ = P.raw_trace(e, "PD", t_on - win - 5, t_on + win + 5)
        v = np.convolve(v, np.ones(50) / 50, mode="same")
        t0 = t_on - win - 5
        rep = np.full(len(c["period"]), np.nan)
        for i in np.r_[pre, post]:
            a = int((c["start"][i] + c["pd_dur"][i] - t0) * 1e4)
            b = int((c["start"][i] + c["period"][i] - t0) * 1e4)
            if b > a:
                rep[i] = (np.argmin(v[a:b])) / 1e4
        ph["repol_a"] = rep
        ph["depol_a"] = c["period"] - c["pd_dur"] - rep
    dper = np.mean(c["period"][post]) - np.mean(c["period"][pre])
    out = {"expt": e, "cond": cond, "onset_min": t_on / 60, "T_onset": c["temp"][k], "n_pre": len(pre), "n_post": len(post),
           "period_pre": np.mean(c["period"][pre]), "period_post": np.mean(c["period"][post]) if len(post) else np.nan, "dperiod_ms": 1e3 * dper}
    for kk, v_ in ph.items():
        out[f"share_{kk}"] = (np.nanmean(v_[post]) - np.nanmean(v_[pre])) / dper if len(post) else np.nan
    return out

rows = [phase_shares(e, CS[e], trace=True) for e in ("969_113", "969_115")]
fa = pd.DataFrame([r for r in rows if r])
print("Intracellular PD traces, Cs+, step 13 -> 15 C; share of the period increase carried by each phase:")
print(fa.round(3).to_string(index=False))
fg = pd.DataFrame([phase_shares(e, CS[e], trace=True, keep_gaps=True) for e in ("969_113", "969_115")])
print("variant, gap cycles kept:")
print(fg[["expt", "n_pre", "n_post", "dperiod_ms", "share_burst", "share_repol_a", "share_depol_a"]].round(3).to_string(index=False))

In [ ]:
intra = ["969_106", "969_113", "969_115", "969_128", "969_152"]
rb = pd.DataFrame([r for r in (phase_shares(e, CS[e]) for e in EXPTS) if r])
rb["intra"] = rb.expt.isin(intra)
cols = ["expt", "intra", "onset_min", "T_onset", "n_pre", "n_post", "dperiod_ms", "share_burst", "share_repol_b", "share_inhib_b", "share_depol_b"]
print("Reading (b), spikes only, Cs+, 13 -> 15 C, every experiment with a step onset found:")
print(rb[cols].round(3).to_string(index=False))
sub = rb[rb.intra & (rb.n_post > 0)]
print(f"\nintracellular experiments with a period increase: {len(sub)}; depolarizing share > 0.5 in {int((sub.share_depol_b > 0.5).sum())}")
allp = rb[rb.n_post > 0]
print(f"all experiments with a period increase: {len(allp)}; depolarizing share > 0.5 in {int((allp.share_depol_b > 0.5).sum())}; "
      f"median shares burst {allp.share_burst.median():.2f}, repol {allp.share_repol_b.median():.2f}, inhib {allp.share_inhib_b.median():.2f}, depol {allp.share_depol_b.median():.2f}")
print("experiments with no cycle longer than the pre-step mean in the first 30 s:", rb[rb.n_post == 0].expt.tolist())
rbg = pd.DataFrame([r for r in (phase_shares(e, CS[e], keep_gaps=True) for e in EXPTS) if r])
sg, ag = rbg[rbg.expt.isin(intra) & (rbg.n_post > 0)], rbg[rbg.n_post > 0]
print(f"variant, gap cycles kept: intracellular {int((sg.share_depol_b > 0.5).sum())} of {len(sg)}, all {int((ag.share_depol_b > 0.5).sum())} of {len(ag)} with depolarizing share > 0.5; "
      f"largest change of a share against the main reading {np.nanmax(np.abs(rbg.set_index('expt')[cols[7:]] - rb.set_index('expt')[cols[7:]]).values):.3f}")
ptx = phase_shares("969_152", "PTX_Cs")
print("\nCs+ + PTX, 969_152, reading (b):", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in ptx.items()} if ptx else None)
d = A["969_152"]; c = d["cycles"]; t_on = ptx["onset_min"] * 60
nlp = int(np.sum((d["LP"] >= t_on - 30) & (d["LP"] < t_on + 30)))
s_idx, e_idx = d["Timings"]["PTX_Cs"]
mw = P.condition_mask(c["start"], d["fileTimes"], s_idx[0], e_idx[0])
lo, hi = c["start"][mw][0], c["start"][mw][-1]
print(f"969_152, Cs+ + PTX: LP spikes within 30 s of the step onset {nlp}; in the whole condition window {int(np.sum((d['LP'] >= lo) & (d['LP'] <= hi)))} "
      f"(saline window: {int(np.sum(~np.isnan(c['LP_ON'][P.condition_mask(c['start'], d['fileTimes'], *[x[0] for x in d['Timings']['Saline']])])))} cycles with an LP onset)")

In [ ]:
# The deposited raw files: channels, and whether a spike file exists for the experiment.
for e in ["969_087", "969_113", "969_115", "969_137", "969_147", "981_007", "981_037"]:
    print(f"Raw_Expt_{e}.mat: channels {P.raw_channels(e)}; spike file {'yes' if f'{e}_SpikeAndTemp.mat' in spike_files else 'no'}")
# Raw files with an intracellular PD channel and no spike file: does the temperature step?
for e in ["981_007"]:
    T_, n_ = P.raw_trace(e, "temp", 0, 1e6)
    Tm = T_[: len(T_) // 300000 * 300000].reshape(-1, 300000).mean(axis=1)
    print(f"Raw_Expt_{e}.mat: {n_ / 6e5:.1f} min; temperature (30 s means) {Tm.min():.2f} to {Tm.max():.2f} C")
    del T_

# Cs+ + PTX example of Figure 4F,G: Raw_Expt_981_037.mat, reading (a) on the warming at 16 min.
t0, t1 = 8 * 60.0, 26 * 60.0
v37, _ = P.raw_trace("981_037", "PD", t0, t1)
T37, _ = P.raw_trace("981_037", "temp", t0, t1)
sp37 = t0 + P.trace_spikes(v37)
st, fr, spb, per, dur, gap = P.extract_bursts(sp37, 0.2, 2, return_gaps=True)
print(f"\n981_037, {t0 / 60:.0f}-{t1 / 60:.0f} min: {len(sp37)} PD spikes detected, {len(st)} bursts, median {np.median(spb):.0f} spikes per burst, "
      f"median frequency {np.median(fr):.2f} Hz; spike peaks {np.percentile(v37[np.round((sp37 - t0) * 1e4).astype(int)], [5, 50, 95]).round(1)} mV "
      f"(5th, 50th, 95th percentile), trace 1st-99th percentile {np.percentile(v37, [1, 99]).round(1)} mV")
ti = t0 + np.arange(len(T37)) / 1e4
csT = np.concatenate([[0.0], np.cumsum(T37)])
a_, b_ = np.searchsorted(ti, st[:-1]), np.searchsorted(ti, st[1:])
c37 = {"temp": (csT[b_] - csT[a_]) / np.maximum(b_ - a_, 1)}
k = P.step_onset(c37, np.ones(len(per), bool), 11, 14)
pre = np.nonzero((st[:-1] >= st[k] - 30) & (st[:-1] < st[k]) & ~gap)[0]
post = np.nonzero((st[:-1] >= st[k]) & (st[:-1] < st[k] + 30) & ~gap)[0]
post = post[per[post] > per[pre].mean()]
vs = np.convolve(v37, np.ones(50) / 50, mode="same")
rep37 = np.full(len(per), np.nan)
for i in np.r_[pre, post]:
    a_, b_ = int((st[i] + dur[i] - t0) * 1e4), int((st[i] + per[i] - t0) * 1e4)
    rep37[i] = np.argmin(vs[a_:b_]) / 1e4
dP = per[post].mean() - per[pre].mean()
print(f"warming onset {st[k] / 60:.2f} min (deposited plotting call 16.06 min) at {c37['temp'][k]:.2f} C, 30 s later {c37['temp'][post[-1]]:.2f} C; "
      f"cycles pre {len(pre)}, post (longer than the pre mean) {len(post)}; period {1e3 * per[pre].mean():.0f} -> {1e3 * per[post].mean():.0f} ms")
for name, x in (("burst", dur), ("repolarization", rep37), ("depolarization", per - dur - rep37)):
    print(f"   share of the period increase, {name}: {(np.nanmean(x[post]) - np.nanmean(x[pre])) / dP:.2f}")


**Result: reproduced.** Under reading (a), on the two Cs+ experiments with an intracellular
PD trace and spike times, the depolarizing phase (voltage minimum to the next burst) carries 1.29
(969_113, the paper's example) and 1.63 (969_115) of the period increase in the first 30 s of
warming from 13 C, the burst 0.05 and 0.02, and the repolarization shrinks (-0.34 and -0.66); with
the gap cycles kept the shares change by at most 0.015. Under reading (b), from spikes, the
depolarizing phase (last LP spike to the next PD burst) carries more than half in 4 of the 5
experiments marked intracellular (0.35 in 969_115, where the repolarization to the first LP spike
carries 0.76), and in 17 of the 20 Cs+ experiments with a period increase (16 of 20 with the gap
cycles kept). The statement's Cs+ + PTX example, outside the criterion: in Raw_Expt_981_037.mat,
the trace the deposited script plots as the PTX example, the period rises from 1628 to 2146 ms in
the first 30 s of the warming at 16.10 min (from 11.3 C), and the depolarizing phase carries 0.96
of the increase, the burst -0.01 and the repolarization 0.06, so that part holds too, on a step
from 11 C, not 13 C, and with spikes detected on the trace. In 969_152 under Cs+ + PTX no LP spike
falls within 30 s of the step onset (the window holds LP spikes elsewhere), so reading (b) cannot
be formed there. No other raw file can add a Cs+ warming with an intracellular PD trace: 969_087
and 969_147 are extracellular only, 969_137 holds PY, and 981_007, whose PD trace the deposited
script plots as the saline and Cs+ example of the paper's Figure 5E, stays between 9.6 and 11.7 C
over its 187 min, so it has no warming step.**

## fig3d-freq-decrease-while-warming and fig3e-freq-increase-when-stable

> Among temperature-increasing cycles, the percent with a frequency decrease was higher in Cs+
> (31.4%) than in saline (7.5%), and higher in Cs+ in 20 of 21 experiments.
>
> Among temperature-stable cycles, the percent with a frequency increase was higher in Cs+ (27.6%)
> than in saline (14.1%), and higher in Cs+ in 20 of 21 experiments.

The jag measure under the O3 readings: centred (MATLAB `smooth`, as the deposited
ProbabilityUpDown.m) or trailing moving average; span `window`, the Timings window (deposited
code), with stable cycles those whose smoothed change is within +-0.01 C (|dT| < 0.01, the deposited
code and the Figure 3A legend, "change less than 0.01 C/pyloric cycle"); span `steps`, the step
protocol from the first 11 C cycle to the end of the 21 C hold (O3 (a)), and span `recording`, the
window extended over the cooling back to 11 C (O3 (b); the Timings windows end at the 21 C hold, and
the cooling comes before the solution change, Methods, Temperature protocol), both with every cycle
that did not warm counted as stable (dT <= 0.01, the Methods and the claims file header). The
deposited code smooths the whole recording, across condition boundaries; `smoothed within` smooths
only the cycles of the window, a variant.

In [ ]:
J = {}
rows = []
for centred, span, within in [(c_, s_, False) for c_ in (True, False) for s_ in ("window", "steps", "recording")] + [(True, "window", True)]:
    v = np.array([[*P.jag_measure(A[e], "Saline", centred, span, smooth_within=within), *P.jag_measure(A[e], CS[e], centred, span, smooth_within=within)] for e in EXPTS])
    J[(centred, span) if not within else "within"] = v
    dec_s, inc_s, dec_c, inc_c = v[:, 0], v[:, 1], v[:, 4], v[:, 5]
    rows.append({"smoothing": ("centred" if centred else "trailing") + (", smoothed within" if within else ""), "span": span,
                 "dec_saline": f"{dec_s.mean():.1f} +- {dec_s.std(ddof=1):.1f}", "dec_Cs": f"{dec_c.mean():.1f} +- {dec_c.std(ddof=1):.1f}",
                 "dec_Cs_higher": int((dec_c > dec_s).sum()), "dec_p": stats.ttest_rel(dec_c, dec_s).pvalue,
                 "inc_saline": f"{inc_s.mean():.1f} +- {inc_s.std(ddof=1):.1f}", "inc_Cs": f"{inc_c.mean():.1f} +- {inc_c.std(ddof=1):.1f}",
                 "inc_Cs_higher": int((inc_c > inc_s).sum()), "inc_p": stats.ttest_rel(inc_c, inc_s).pvalue,
                 "warming_cycles": f"{int(v[:, 2].min())}-{int(v[:, 2].max())}", "stable_cycles": f"{int(v[:, 3].min())}-{int(v[:, 3].max())}"})
print(pd.DataFrame(rows).to_string(index=False))
SW = P.load_mat("Sawtooth.mat")
dep_dec, dep_inc = 100 * np.asarray(SW["DecFreq"]), 100 * np.asarray(SW["IncFreqTempZCnt"])
sw_files = [str(np.ravel(x)[0]) for x in np.ravel(SW["files"]["name"])]
order = [sw_files.index(e) for e in EXPTS]
v = J[(True, "window")]
print(f"\ndeposited Sawtooth.mat: dec saline {dep_dec[0].mean():.1f}, Cs {dep_dec[1].mean():.1f}; inc saline {dep_inc[0].mean():.1f}, Cs {dep_inc[1].mean():.1f}")
print("per experiment, centred/window here against deposited (percent):")
print(pd.DataFrame({"dec_sal": v[:, 0], "dep": dep_dec[0, order], "dec_cs": v[:, 4], "dep ": dep_dec[1, order],
                    "inc_sal": v[:, 1], "dep  ": dep_inc[0, order], "inc_cs": v[:, 5], "dep   ": dep_inc[1, order]}, index=EXPTS).round(1).T.to_string())

**Result: reproduced (both claims).** Under the centred window reading (the deposited code's), the
percent of warming cycles with a frequency decrease is 6.3 +- 4.8 in saline and 31.0 +- 15.7 in Cs+,
higher in Cs+ in 20 of 21, paired t p = 4e-07; the percent of stable cycles with an increase is
13.2 +- 5.4 and 26.9 +- 6.2, higher in Cs+ in 20 of 21, p = 3e-09. All seven readings (centred and
trailing; the window, the step protocol, and the window with the cooling, O3 (b); and smoothing
within the window) give means within the tolerances, 20 of 21 and p < 2e-05. Per experiment the
values are close to the deposited Sawtooth.mat.**

## fig4c-ptx-restores-frequency-change, fig4d-ptx-keeps-decrease-while-warming, fig4e-stable-increase-ordering

> The frequency change from 11 to 21 degrees C differed by condition; it was smaller in Cs+ than in
> saline or in Cs+ + PTX, with no difference between saline and Cs+ + PTX.
>
> The percent of temperature-increasing cycles with a frequency decrease differed by condition; it
> was higher in Cs+ and in Cs+ + PTX than in saline, with no difference between Cs+ and Cs+ + PTX.
>
> The percent of temperature-stable cycles with a frequency increase differed between all three
> conditions, lowest in saline, middle in Cs+ and highest in Cs+ + PTX.

The picrotoxin experiments are the paper's six: 969_147, 969_150, 969_152, 969_154, and the two
preparations of 981_035 (its `PD` and `PD2` trains, two rhythms recorded with one thermocouple,
data check above). For
969_147 the Cs+ protocol is `Cs_1` (it matches the deposited summary; `Cs_2` is a second, shorter Cs+
window). 981_035 never came within 0.3 C of 21 C (its top hold is at 20.7 C), so with the 0.3 C rule
it has no 21 C value; the deposited summary has one, so a variant uses 0.5 C for the 21 C hold.
`gaps kept` is the frequency change with the cycles left by the outlier removal kept, as the
deposited code keeps them.

In [ ]:
PTX6 = PTX_EXPTS + ["981_035_PD2"]
CS_PTX = {e: ("Cs_1" if e == "969_147" else "Cs") for e in PTX6}
conds = lambda e: ["Saline", CS_PTX[e], "PTX_Cs"]
def dfreq(e, moe_top, drop_gaps=True):
    out = []
    for i, cnd in enumerate(conds(e)):
        h = P.hold_means(A[e], cnd, which="first", strict_cs=i > 0, drop_gaps=drop_gaps)
        top = P.hold_means(A[e], cnd, which="all", strict_cs=i > 0, moe=moe_top, drop_gaps=drop_gaps)
        out.append(top["freq"][-1] - h["freq"][0])
    return out
names3 = ["saline", "Cs", "Cs+PTX"]
def report(Y, label):
    F, d1, d2, p, _, n = P.rm_anova(Y)
    print(f"  {label}: n {n}, means {dict(zip(names3, np.nanmean(Y[~np.isnan(Y).any(1)], 0).round(3)))}, F({d1},{d2}) = {F:.2f}, p = {p:.3g}")
    for t in P.tukey_rm(Y, names3):
        print(f"     {t['pair']:18s} diff {t['diff']:+.3f}  p multcompare {t['p_multcompare']:.3f}  p classic {t['p_classic']:.3f}")
for moe_top in (0.3, 0.5):
    Y6 = np.array([dfreq(e, moe_top) for e in PTX6])
    print(f"21 C hold within {moe_top} C:")
    print(pd.DataFrame(Y6, index=PTX6, columns=names3).round(3).to_string())
    report(Y6, "six (paper's set)")
    report(np.array([dfreq(e, moe_top, drop_gaps=False) for e in PTX6]), "six, gaps kept")
SPt = np.c_[np.asarray(SP["TotalSals"])[:, -1] - np.asarray(SP["TotalSals"])[:, 0], np.asarray(SP["TotalExpts"])[:, -1] - np.asarray(SP["TotalExpts"])[:, 0],
            np.asarray(SP["TotalExpts1"])[:, -1] - np.asarray(SP["TotalExpts1"])[:, 0]]
print("deposited SummaryStatisticsPTX:")
report(SPt, "six, deposited values")

In [ ]:
JP = {}
for centred in (True, False):
    for span in ("window", "steps", "recording"):
        JP[(centred, span)] = np.array([[P.jag_measure(A[e], cnd, centred, span)[:2] for cnd in conds(e)] for e in PTX6])
for centred, span in JP:
    v = JP[(centred, span)]
    print(f"[{'centred' if centred else 'trailing'}, {span}] decrease while warming (percent):")
    print(pd.DataFrame(v[:, :, 0], index=PTX6, columns=names3).round(1).T.to_string())
    report(v[:, :, 0], "six")
dp = 100 * np.asarray(SWP["DecFreq"]).T
print("deposited SawtoothPTX DecFreq (percent), columns as its file list plus a sixth:")
print(dp.round(1).T)
report(dp, "six, deposited values")

In [ ]:
for centred, span in JP:
    v = JP[(centred, span)]
    print(f"[{'centred' if centred else 'trailing'}, {span}] increase while stable (percent):")
    print(pd.DataFrame(v[:, :, 1], index=PTX6, columns=names3).round(1).T.to_string())
    report(v[:, :, 1], "six")
ip = 100 * np.asarray(SWP["IncFreqTempZCnt"]).T
report(ip, "six, deposited values")

**Result: fig4c partially reproduced, resting on an assumption; fig4d and fig4e reproduced.**
Frequency change (fig4c): with the criterion's 0.3 C rule the two preparations of 981_035 have no
21 C value, so only four experiments are complete and the condition effect is not significant
(F(2,6) = 4.21, p = 0.072; no pair is significant, p >= 0.06). The paper's F(2,10) shows it entered
both; taking their top hold with a 0.5 C margin (an assumption), the six give F(2,10) = 10.43,
p = 0.0036, saline 0.92, Cs+ 0.38 and Cs+ + PTX 0.73 Hz. Cs+ is below saline under both Tukey readings
(p = 0.025 and 0.003), and saline against Cs+ + PTX is not significant (p = 0.114 and 0.307), but
Cs+ is below Cs+ + PTX only under the classic Tukey (p = 0.036), not under the multcompare reading
of the deposited code (p = 0.106). The full pattern thus holds only with a margin wider than the
criterion's and under one of the two post hoc readings; the Cs+ reduction against saline holds
there under both. Keeping the gap cycles gives nearly the same values (F(2,10) = 10.38, the same pattern). Decrease while warming (fig4d),
centred window, six: F(2,10) = 13.00, p = 0.0017, saline 2.0%, Cs+ 28.9%, Cs+ + PTX 30.8%; both Cs+
conditions above saline (p <= 0.023), Cs+ against Cs+ + PTX not significant (p >= 0.95); all six
readings agree (condition p <= 0.002, both Cs+ conditions above saline at p <= 0.031, Cs+ against
Cs+ + PTX p >= 0.87). Increase while stable (fig4e): F(2,10) = 79.5, p = 7e-07, saline 7.5% < Cs+
24.6% < Cs+ + PTX 38.0%, all three pairs significant under both Tukey readings (p <= 0.012); all six
readings agree (pairs p <= 0.015).**

## Phases at 11 C: fig5b, Table 1, fig5d, LP duty cycle

> fig5b-pd-burst-shorter: At 11 degrees C the PD burst was modestly shorter in Cs+ (144 ms) than in
> saline (164 ms).
>
> table1-pd-off-advanced: PD OFF phase advanced in Cs+ (0.12) compared with saline (0.19).
> table1-lp-on-advanced: LP ON 0.29 against 0.40. table1-lp-off-advanced: LP OFF 0.56 against 0.66.
> table1-py-phase-unchanged: PY ON (0.58 against 0.60) and PY OFF (0.87 against 0.90) did not differ.
>
> fig5d-pdoff-lpon-delay-smaller: The phase delay between PD OFF and LP ON was smaller in Cs+ (0.17)
> than in saline (0.21).
>
> lp-duty-cycle-preserved: LP duty cycle did not differ between saline and Cs+ at 11 degrees C.

Per experiment, means over the 11 C cycles: O4 (a), cycles within 0.3 C of 11 C in the first hold
(O2); O4 (b), every cycle of the first holding period, from the first cycle within 1 C of 11 C (a
standard bound) to the onset of the warming toward 13 C, whatever its temperature. The onset is
traced back, by the paper's increasing-temperature rule, from the rise that leaves the band within
0.3 C of 11 C for the last time before 12.7 C is reached, so that a brief rise during the hold does
not end it. `a_gaps` is
(a) with the cycles left by the outlier removal kept, as the deposited code keeps them (data
check, above). Onsets and offsets as the deposited Phase_Calc.m (the
longest run of spikes with gaps of at most 0.11 s in the cycle), which is O6 with the burst rule
taken from the paper's code. The deposited code's PD burst duration is period times PD OFF phase
(MakePaperFigures_Clean.m, Fig 6B); it is shown as a variant.

In [ ]:
def first_hold_cycles(d, cond, reading):
    """Cycles of the first 11 C hold. (a) within 0.3 C of 11 C (O2, as for frequency); (b) every
    cycle of the holding period, from the first cycle of the condition window within 1 C of 11 C
    (a standard bound) to the last cycle before the warming toward 13 C begins (step_onset with
    hold_end: the rise from the last cycle within 0.3 C of 11 C, traced back by the paper's
    increasing-temperature rule), whatever its temperature; (a_gaps) as (a) with the cycles
    left by the outlier removal kept, as the deposited code keeps them."""
    h = P.hold_means(d, cond, which="first", strict_cs=False, drop_gaps=reading != "a_gaps")
    ii = h["cycle_index"][11]
    if reading in ("a", "a_gaps"):
        return ii
    c = d["cycles"]
    s_idx, e_idx = d["Timings"][cond]
    m = P.condition_mask(c["start"], d["fileTimes"], s_idx[0], e_idx[0]) & (c["freq"] > 1 / 8)
    k = P.step_onset(c, m, 11, 13, hold_end=True)
    idx = np.nonzero(m)[0]
    near = idx[np.abs(c["temp"][idx] - 11) <= 1]
    if k is None or not len(near):
        return np.array([], int)
    jj = idx[(idx >= near[0]) & (idx < k)]
    return jj[~c["gap"][jj]]

def per_expt(reading, expts=EXPTS):
    rows = []
    for e in expts:
        d = A[e]
        for cond, lab in (("Saline", "S"), (CS.get(e, "Cs"), "C")):
            ii = first_hold_cycles(d, cond, reading)
            c = d["cycles"]
            r = {"expt": e, "cond": lab, "n": len(ii), "T_min": np.min(c["temp"][ii]) if len(ii) else np.nan, "T_max": np.max(c["temp"][ii]) if len(ii) else np.nan}
            for k in ("freq", "period", "pd_dur", "PD_OFF", "LP_ON", "LP_OFF", "PY_ON", "PY_OFF", "LP_duty"):
                r[k] = np.nanmean(c[k][ii]) if len(ii) else np.nan
            r["delay"] = np.nanmean((c["LP_ON"] - c["PD_OFF"])[ii]) if len(ii) else np.nan
            rows.append(r)
    return pd.DataFrame(rows).pivot(index="expt", columns="cond")

PH = {r_: per_expt(r_) for r_ in ("a", "b", "a_gaps")}
print("O4 (a), per experiment (S saline, C Cs+):")
print(PH["a"][["n", "pd_dur", "PD_OFF", "LP_ON", "LP_OFF", "PY_ON", "PY_OFF"]].round(3).to_string())
print("\nO4 (b), cycles of the whole first holding period: number and temperature range per experiment")
print(PH["b"][["n", "T_min", "T_max"]].round(2).to_string())
onsets = []
for e in EXPTS:
    for cond, lab in (("Saline", "S"), (CS[e], "C")):
        d = A[e]; c = d["cycles"]; s_idx, e_idx = d["Timings"][cond]
        k = P.step_onset(c, P.condition_mask(c["start"], d["fileTimes"], s_idx[0], e_idx[0]) & (c["freq"] > 1 / 8), 11, 13, hold_end=True)
        ia = first_hold_cycles(d, cond, "a")
        onsets.append({"expt": e, "cond": lab, "onset_min": c["start"][k] / 60 if k is not None else np.nan, "T_onset": c["temp"][k] if k is not None else np.nan,
                       "n_a": PH["a"].loc[e, ("n", lab)], "n_b": PH["b"].loc[e, ("n", lab)],
                       "n_a_from_onset": int(np.sum(ia >= k)) if k is not None else 0})
on = pd.DataFrame(onsets)
print("\nOnset of the warming toward 13 C, and the cycles of each reading; experiments where (b) has fewer cycles than (a)\n"
      "(n_a_from_onset: cycles of (a) from the onset on, in the first part of the warming but still within 0.3 C of 11 C):")
print(on[on.n_b < on.n_a].round(2).to_string(index=False))
print(f"cycles per experiment, (a) {int(PH['a']['n'].min().min())}-{int(PH['a']['n'].max().max())}, (b) {int(PH['b']['n'].min().min())}-{int(PH['b']['n'].max().max())}")

def cmp(k, reading, scale=1.0):
    x, y = PH[reading][(k, "S")] * scale, PH[reading][(k, "C")] * scale
    r = P.paired(x, y)
    return r


In [ ]:
for r_ in ("a", "b", "a_gaps"):
    r = cmp("pd_dur", r_, 1000)
    print(f"[{r_}] PD burst (first to last spike): saline {r['mean_x']:.0f} +- {r['sd_x']:.0f} ms, Cs {r['mean_y']:.0f} +- {r['sd_y']:.0f} ms, n {r['n']}, "
          f"Wilcoxon p = {r['p_w']:.3g}, paired t p = {r['p_t']:.3g}; Cs shorter in {int((PH[r_][('pd_dur', 'C')] < PH[r_][('pd_dur', 'S')]).sum())}")
    xs = PH[r_][("period", "S")] * PH[r_][("PD_OFF", "S")] * 1000; xc = PH[r_][("period", "C")] * PH[r_][("PD_OFF", "C")] * 1000
    r = P.paired(xs, xc)
    print(f"    variant, period x PD OFF (deposited code): saline {r['mean_x']:.0f} +- {r['sd_x']:.0f}, Cs {r['mean_y']:.0f} +- {r['sd_y']:.0f} ms, Wilcoxon p = {r['p_w']:.3g}")

**Result: reproduced.** Over the 20 experiments with an 11 C value in both conditions, PD
burst duration is 164 +- 36 ms in saline and 145 +- 19 ms in Cs+, within the tolerances, Wilcoxon
p = 0.012, shorter in Cs+ in 15. Reading (b) (165 and 146 ms, p = 0.014), the deposited code's
measure (period times PD OFF, 164 and 145 ms, p = 0.017) and the gap cycles kept (164 and 145 ms,
p = 0.012) agree. Reading (b) holds fewer cycles than reading (a) only by the cycles of (a) that
lie in the first part of the warming, after its onset but still within 0.3 C of 11 C (3 to 23 per
experiment, printed above).**

In [ ]:
for k in ("PD_OFF", "LP_ON", "LP_OFF"):
    for r_ in ("a", "b", "a_gaps"):
        r = cmp(k, r_)
        print(f"[{r_}] {k}: saline {r['mean_x']:.3f} +- {r['sd_x']:.3f}, Cs {r['mean_y']:.3f} +- {r['sd_y']:.3f}, n {r['n']}, paired t = {r['t']:.2f}, p = {r['p_t']:.2g} "
              f"(Wilcoxon {r['p_w']:.2g}; Shapiro-Wilk on the differences p = {r['sw_diff']:.2f}); Cs lower in {int((PH[r_][(k, 'C')] < PH[r_][(k, 'S')]).sum())}")

**Result: reproduced (three claims).** At 11 C, over 20 experiments, PD OFF is 0.191 +- 0.044
in saline and 0.122 +- 0.027 in Cs+ (paired t p = 2e-07), LP ON 0.398 +- 0.049 and 0.289 +- 0.051
(p = 2e-07), LP OFF 0.664 +- 0.053 and 0.565 +- 0.056 (p = 1e-07), all within the tolerances and
lower in Cs+ in 20, 19 and 20 experiments; reading (b) (0.191 and 0.123, 0.398 and 0.290, 0.664 and
0.565, 20 experiments each) and the gap cycles kept (0.198 and 0.126, 0.404 and 0.295, 0.671 and
0.571) agree.**

In [ ]:
for r_ in ("a", "b", "a_gaps"):
    t = PH[r_]
    have = t[("PY_ON", "S")].notna() & t[("PY_ON", "C")].notna()
    print(f"[{r_}] experiments with PY phases in both conditions: {int(have.sum())} ({t.index[have].tolist()})")
    for k, test in (("PY_ON", "p_t"), ("PY_OFF", "p_w")):
        r = cmp(k, r_)
        print(f"    {k}: saline {r['mean_x']:.3f} +- {r['sd_x']:.3f}, Cs {r['mean_y']:.3f} +- {r['sd_y']:.3f}, mean difference {r['mean_diff']:+.3f}, n {r['n']}, "
              f"paper's test ({'paired t' if test == 'p_t' else 'Wilcoxon'}) p = {r[test]:.3f} (other test {r['p_w' if test == 'p_t' else 'p_t']:.3f})")

**Result: reproduced.** Over the 12 experiments with PY phases in both conditions (the
paper's 12), PY ON is 0.603 against 0.598 (paired t p = 0.85, difference 0.006) and PY OFF 0.884
against 0.897 (Wilcoxon p = 0.68, difference -0.013), within the tolerances; reading (b) (p = 0.85
and 0.97) and the gap cycles kept (p = 0.73 and 0.42) agree.**

In [ ]:
for r_ in ("a", "b", "a_gaps"):
    r = cmp("delay", r_)
    v = P.paired(PH[r_][("LP_ON", "S")] - PH[r_][("PD_OFF", "S")], PH[r_][("LP_ON", "C")] - PH[r_][("PD_OFF", "C")])
    print(f"[{r_}] LP ON - PD OFF per cycle: saline {r['mean_x']:.3f} +- {r['sd_x']:.3f}, Cs {r['mean_y']:.3f} +- {r['sd_y']:.3f}, n {r['n']}, paired t p = {r['p_t']:.3g}; "
          f"difference of the means: saline {v['mean_x']:.3f}, Cs {v['mean_y']:.3f}, p = {v['p_t']:.3g}")

**Result: reproduced.** LP ON minus PD OFF is 0.207 +- 0.064 in saline and 0.167 +- 0.044 in Cs+
(20 experiments), within the tolerances, paired t p = 0.004; reading (b) (p = 0.004), the difference
of the means and the gap cycles kept (p = 0.008) agree.**

In [ ]:
for r_ in ("a", "b", "a_gaps"):
    r = cmp("LP_duty", r_)
    print(f"[{r_}] LP duty cycle: saline {r['mean_x']:.3f} +- {r['sd_x']:.3f}, Cs {r['mean_y']:.3f} +- {r['sd_y']:.3f}, mean difference {r['mean_diff']:+.3f}, "
          f"n {r['n']}, paired t p = {r['p_t']:.3f}")

**Result: reproduced.** LP duty cycle is 0.267 in saline and 0.276 in Cs+ (difference -0.009,
20 experiments); the paired t-test does not find a difference, p = 0.445 (reading (b) 0.433, gap
cycles kept 0.449).**

## matched-period-lp-earlier

> In cycles of matched period between saline at 11 degrees C and Cs+, PD bursts were 51 ms shorter
> in Cs+ while LP fired 91 ms earlier and PY latency did not change (1.9 ms), and the PD offset
> advance did not correlate with the LP onset advance.

Saline reference: the first 11 C hold (O2, O4 (a)). Readings: (a) the Cs+ holding temperature whose
mean period is nearest the saline 11 C period; (b) Cs+ cycles of the window with a period within 2%
of it. Latencies from the first PD spike; advance is saline minus Cs+. PD offset advance in ms is
the PD burst shortening (the PD offset latency is the burst duration). The deposited
LPAdvanceSamePeriod.mat, made by code that is not deposited, on a different set of 21 (it adds
969_120, which has no spike file, and 969_154, and lacks 965_106 and 981_010), is shown as a check.

In [ ]:
def matched(e, reading, keep_gaps=False):
    d = A[e]; c = d["cycles"]
    ref = first_hold_cycles(d, "Saline", "a_gaps" if keep_gaps else "a")
    if not len(ref):
        return None
    P0 = np.mean(c["period"][ref])
    s_idx, e_idx = d["Timings"][CS[e]]
    m = P.condition_mask(c["start"], d["fileTimes"], s_idx[0], e_idx[0]) & (c["freq"] > 1 / 8) & (keep_gaps | ~c["gap"])
    if reading == "a":
        h = P.hold_means(d, CS[e], strict_cs=True, drop_gaps=not keep_gaps)
        j = int(np.nanargmin(np.abs(h["period"] - P0)))
        jj = h["cycle_index"][int(P.TARGETS[j])]
        lab = int(P.TARGETS[j])
    else:
        jj = np.nonzero(m & (np.abs(c["period"] - P0) <= 0.02 * P0))[0]
        lab = len(jj)
    if not len(jj):
        return None
    f = lambda k, ii: np.nanmean(c[k][ii]) * 1000
    return {"expt": e, "match": lab, "period_sal_ms": P0 * 1000, "period_cs_ms": np.mean(c["period"][jj]) * 1000,
            "PD_short_ms": f("pd_dur", ref) - f("pd_dur", jj), "LP_adv_ms": f("LP_lat_on", ref) - f("LP_lat_on", jj),
            "PY_adv_ms": f("PY_lat_on", ref) - f("PY_lat_on", jj)}

for r_, kg in (("a", False), ("b", False), ("a", True), ("b", True)):
    t = pd.DataFrame([x for x in (matched(e, r_, kg) for e in EXPTS) if x])
    print(f"reading ({r_}){', variant with the gap cycles kept' if kg else ''}; match = {'Cs+ holding temperature' if r_ == 'a' else 'number of Cs+ cycles'}")
    if not kg:
        print(t.round(1).to_string(index=False))
    else:
        print(f"  LP advance larger than PD shortening: {t.LP_adv_ms.mean() > t.PD_short_ms.mean()}")
    ok = t.PD_short_ms.notna() & t.LP_adv_ms.notna()
    rr = stats.pearsonr(t.PD_short_ms[ok], t.LP_adv_ms[ok])
    print(f"  PD shortening {t.PD_short_ms.mean():.0f} +- {t.PD_short_ms.std():.0f} ms, LP advance {t.LP_adv_ms.mean():.0f} +- {t.LP_adv_ms.std():.0f} ms, "
          f"PY advance {t.PY_adv_ms.mean():.1f} +- {t.PY_adv_ms.std():.0f} ms (n {t.PY_adv_ms.notna().sum()}); "
          f"Pearson r = {rr.statistic:.2f}, p = {rr.pvalue:.3f}, n pairs {int(ok.sum())}\n")
L = P.load_mat("LPAdvanceSamePeriod.mat")
cPD = 1000 * (np.asarray(L["meanPDDUR"])[0] - np.asarray(L["meanPDDUR"])[1])
cLP = 1000 * (np.asarray(L["meanLPDelay"])[0] - np.asarray(L["meanLPDelay"])[1])
ok = ~np.isnan(cPD) & ~np.isnan(cLP)
rr = stats.pearsonr(cPD[ok], cLP[ok])
pyo = np.asarray(L["meanPYON"]) / np.asarray(L["meanFreq"])
print(f"deposited LPAdvanceSamePeriod: PD shortening {np.mean(cPD[ok]):.0f} +- {np.std(cPD[ok], ddof=1):.0f} ms, LP {np.mean(cLP[ok]):.0f} +- {np.std(cLP[ok], ddof=1):.0f} ms, "
      f"PY {1000 * np.nanmean(pyo[0] - pyo[1]):.1f} ms (n {int(np.sum(~np.isnan(pyo[0] - pyo[1])))}); r = {rr.statistic:.2f}, p = {rr.pvalue:.4f}, n pairs {int(ok.sum())}")
print("files:", [str(np.ravel(x)[0]) for x in np.ravel(L["files"]["name"])])

**Result: reproduced under both readings.** With Cs+ cycles within 2% of the saline 11 C period
(reading (b), 19 experiments), PD bursts are 63 +- 35 ms shorter in Cs+, LP fires 95 +- 71 ms
earlier and PY 16 +- 110 ms earlier (11 experiments with PY), all within the tolerances, LP advance
exceeds PD shortening, and the correlation of the two is not significant (r = 0.18, p = 0.46, 19
pairs). Under reading (a), the Cs+ holding temperature nearest in period (21 experiments), PD
shortening is 50 +- 34 ms, LP advance 71 +- 65 ms and PY -33 +- 110 ms, within the tolerances, LP
advance exceeds PD shortening, and r = 0.10, p = 0.68. With the gap cycles kept, reading (b) gives the
same (63 and 96 ms, r = 0.10, p = 0.69) and reading (a) fails (LP advance 39 ms, below the PD
shortening of 51 ms; r = 0.50, p = 0.020), so the outcome does not rest on leaving them out. The deposited LPAdvanceSamePeriod.mat gives
the paper's numbers exactly (51 +- 30, 91 +- 68, 1.9 ms, r = 0.26, p = 0.2928 over 18 pairs), on its
own set of experiments.**

## Phase constancy: fig6a-lp-off-advances-saline, fig6a-lp-advances-cs, fig6a-pd-py-phase-constant

> In saline, LP OFF phase advanced modestly with temperature, by 0.07 per 10 degrees C; all other
> phases were constant.
>
> In Cs+, LP ON and LP OFF advanced with temperature, by 0.05 and 0.14 per 10 degrees C.
>
> PD and PY phases were constant with temperature in both saline and Cs+.

Per experiment and condition a line of phase on temperature: O4 (a) through the six per-temperature
means against 11, 13, ..., 21 (the deposited code, Fig 7A stats); O4 (b) through the phase of every
cycle of the condition window against its temperature ("all cycles", transitions included); the
variant `b_hold` through the cycles within 0.3 C of a target only; `a_gaps` is (a) with the cycles
left by the outlier removal kept, as the deposited code keeps them. Advance per 10 C is minus
ten times the slope. Tests against zero across experiments, p times 10 (Bonferroni, Methods); the
test is chosen by Shapiro-Wilk on the slopes (p < 0.05: Wilcoxon), and both are printed.

In [ ]:
def slopes(reading):
    """(a) line through the six hold means against 11, 13, ..., 21; (b) through every cycle of the
    condition window against its temperature; (b_hold) through the cycles within 0.3 C of a target;
    (a_gaps) as (a) with the cycles left by the outlier removal kept, as the deposited code."""
    out = {}
    for e in EXPTS:
        d = A[e]; c = d["cycles"]
        for cond, lab in (("Saline", "S"), (CS[e], "C")):
            h = P.hold_means(d, cond, strict_cs=False, drop_gaps=reading != "a_gaps")
            s_idx, e_idx = d["Timings"][cond]
            win = P.condition_mask(c["start"], d["fileTimes"], s_idx[0], e_idx[0]) & (c["freq"] > 1 / 8) & ~c["gap"]
            for k in ("PD_OFF", "LP_ON", "LP_OFF", "PY_ON", "PY_OFF"):
                if reading in ("a", "a_gaps"):
                    y = h[k]; x = P.TARGETS.astype(float)
                else:
                    ii = np.concatenate([h["cycle_index"][int(T)] for T in P.TARGETS]) if reading == "b_hold" else np.nonzero(win)[0]
                    y = c[k][ii]; x = c["temp"][ii]
                ok = ~np.isnan(y) & ~np.isnan(x)
                out[(e, lab, k)] = np.polyfit(x[ok], y[ok], 1)[0] if ok.sum() >= 2 and np.ptp(x[ok]) > 0 else np.nan
    s = pd.Series(out).unstack([1, 2])
    return s

READINGS6 = ("a", "b", "b_hold", "a_gaps")
SL = {r_: slopes(r_) for r_ in READINGS6}
rows = []
for r_ in READINGS6:
    for lab in ("S", "C"):
        for k in ("PD_OFF", "LP_ON", "LP_OFF", "PY_ON", "PY_OFF"):
            v = SL[r_][(lab, k)].dropna().values
            sw = stats.shapiro(v).pvalue
            pw = stats.wilcoxon(v).pvalue
            pt = stats.ttest_1samp(v, 0).pvalue
            chosen = pw if sw < 0.05 else pt
            rows.append({"reading": r_, "cond": "saline" if lab == "S" else "Cs", "phase": k, "n": len(v), "advance_per_10C": -10 * v.mean(),
                         "sd": 10 * v.std(ddof=1), "shapiro_p": sw, "test": "Wilcoxon" if sw < 0.05 else "t", "p_adj_chosen": min(1, 10 * chosen),
                         "p_adj_wilcoxon": min(1, 10 * pw), "p_adj_t": min(1, 10 * pt)})
FT = pd.DataFrame(rows)
print(FT.round(4).to_string(index=False))


In [ ]:
# The same slope tests on the deposited per-experiment phases of SummaryStatistics.mat (TotalPDOFF_S
# ... TotalPYOFF_E, columns 3 to 8 in MATLAB), as the deposited MakePaperFigures_Clean.m runs them
# ("Stats for Fig7A"): a line against 11, 13, ..., 21 per experiment, PY only where the saline PY ON
# row is complete, the test fixed by its NormalityCheck (2 = signed-rank), p times 10.
NORM_DEP = [1, 1, 2, 1, 2, 1, 2, 2, 1, 2]
xT = P.TARGETS.astype(float)
rows = []
for j, (cnd, lab) in enumerate((("S", "saline"), ("E", "Cs"))):
    for i, k in enumerate(("PDOFF", "LPON", "LPOFF", "PYON", "PYOFF")):
        Y = np.asarray(S[f"Total{k}_{cnd}"])[:, 2:8].astype(float)
        py_ok = ~np.isnan(np.asarray(S["TotalPYON_S"])[:, 2:8].astype(float)).any(axis=1)
        v = []
        for r, y in enumerate(Y):
            ok = ~np.isnan(y)
            if (k.startswith("PY") and not py_ok[r]) or ok.sum() < 2:
                continue
            v.append(np.polyfit(xT[ok], y[ok], 1)[0])
        v = np.array(v)
        pw, pt = stats.wilcoxon(v).pvalue, stats.ttest_1samp(v, 0).pvalue
        rows.append({"cond": lab, "phase": k, "n": len(v), "advance_per_10C": -10 * v.mean(), "sd": 10 * v.std(ddof=1),
                     "shapiro_p": stats.shapiro(v).pvalue, "deposited_test": "Wilcoxon" if NORM_DEP[5 * j + i] == 2 else "t",
                     "p_adj_deposited_test": min(1, 10 * (pw if NORM_DEP[5 * j + i] == 2 else pt)), "p_adj_wilcoxon": min(1, 10 * pw), "p_adj_t": min(1, 10 * pt)})
print("Slopes fitted to the deposited per-experiment phases (SummaryStatistics.mat):")
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: all three reproduced.** Under O4 (a), in saline LP OFF advances 0.071 +- 0.065 per 10 C,
Wilcoxon adjusted p = 0.0001, with a negative slope (reading (b), every cycle of the window, 0.064,
t-test adjusted p = 0.0005). In Cs+, LP ON advances 0.053 +- 0.052 per 10 C (Wilcoxon adjusted
p = 0.0003) and LP OFF 0.140 +- 0.096 (t-test, p < 0.0001), both within the tolerances and below
0.01; readings (b) (0.047 and 0.122, t-test adjusted p = 0.0020 and < 0.0001, the test Shapiro-Wilk
chooses; Wilcoxon 0.0005 and < 0.0001) and `b_hold` agree. With the gap cycles kept, as the
deposited code, LP ON in Cs+ advances 0.010 +- 0.202 with adjusted p = 0.029: the gap cycles,
whose phases are divided by the wrong period, widen the slopes' spread about fourfold, and only
under that variant does LP ON miss p < 0.01. The authors' own per-experiment phases
(SummaryStatistics.mat), fitted and tested as the deposited script does, give what leaving the gap
cycles out gives: LP ON in Cs+ advances 0.054 +- 0.055 per 10 C with Wilcoxon adjusted p = 0.0004,
LP OFF in Cs+ 0.140 +- 0.099 (p < 0.0001) and LP OFF in saline 0.074 +- 0.075 (p = 0.0002), with
PD OFF, PY ON, PY OFF and saline LP ON at adjusted p >= 0.056. PD OFF, PY ON and PY OFF have
adjusted p >= 0.08 in both conditions under every reading here (the test chosen by Shapiro-Wilk),
and LP ON in saline 0.35: none is significant. PY slopes use 13 experiments in saline (969_088 has
PY in saline only) and 12 in Cs+; the deposited script keeps the 11 experiments with a complete
saline PY ON row.**

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| figs1-ih-reduced-11c | blocked (data missing) |
| figs1-ih-reduced-21c | blocked (data missing) |
| fig1-cs-lower-frequency-11c | reproduced |
| fig1d-difference-grows-with-temperature | reproduced |
| fig1e-q10-lower-in-cs | reproduced |
| fig1f-two-saline-frequency-change-same | reproduced (on the deposited per-experiment values) |
| fig1e-two-saline-q10-same | blocked (data missing) |
| fig1e-saline-q10-same-across-designs | blocked (data missing) |
| fig2c-decrease-by-depolarizing-phase | reproduced (the Cs+ + PTX example, outside the criterion, agrees) |
| fig3d-freq-decrease-while-warming | reproduced |
| fig3e-freq-increase-when-stable | reproduced |
| fig4c-ptx-restores-frequency-change | partially reproduced, resting on an assumption |
| fig4d-ptx-keeps-decrease-while-warming | reproduced |
| fig4e-stable-increase-ordering | reproduced |
| fig5b-pd-burst-shorter | reproduced |
| table1-pd-off-advanced | reproduced |
| table1-lp-on-advanced | reproduced |
| table1-lp-off-advanced | reproduced |
| table1-py-phase-unchanged | reproduced |
| fig5d-pdoff-lpon-delay-smaller | reproduced |
| lp-duty-cycle-preserved | reproduced |
| matched-period-lp-earlier | reproduced |
| fig6a-lp-off-advances-saline | reproduced |
| fig6a-lp-advances-cs | reproduced, resting on an assumption |
| fig6a-pd-py-phase-constant | reproduced |

**How the analysis differs from the paper's.** Everything is computed in Python from the deposited
spike and temperature files with a port of the paper's deposited MATLAB code (burst detection,
phase, cycle temperature, condition windows, the 1/8 Hz cycle filter, the jag measure); the
per-experiment values agree with the deposited summary files to a few thousandths in most cases,
not exactly. The cycles that the deposited outlier removal leaves spanning two bursts (1.6%) are
left out of the hold means and phases, an assumption that brings the phases much closer to the
deposited summaries and that the authors' own per-experiment phases support for the slopes of
fig6a (their LP ON slope in Cs+ has adjusted p = 0.0004). Every claim built on hold means or phases
prints the variant with them kept (the jag measure of fig3d,e and fig4d,e counts every cycle's
frequency, which these cycles keep): it changes no outcome but fig6a-lp-advances-cs, whose LP ON slope then misses
p < 0.01; matched-period-lp-earlier then holds under its reading (b) only. The 11 C values are from the first 11 C hold, which is every 11 C cycle of the
deposited windows. Repeated-measures ANOVA and Tukey comparisons are computed here (two Tukey
readings). The picrotoxin tests use the paper's six, which include the two preparations recorded in
981_035, and take their top hold (20.7 C) as the 21 C value, which the criterion's 0.3 C rule does
not allow; the two-saline frequency test uses the deposited per-experiment table. The Cs+ + PTX
example of fig2c is analysed on PD spikes detected on its intracellular trace. No step uses random
draws.

**What was found about the dataset.** (1) VoltageClampData.mat holds only the first trial of each
experiment, no Cs+ trial, while its ConditionsKey lists all of them: the Ih numbers cannot be
recomputed. (2) Five of the six two-saline control experiments have no spike file; ControlTable
holds their per-experiment values only, and says three of their second protocols were ramps.
(3) 981_035 holds two PD spike trains (`PD`, `PD2`, not in the README) that do not burst
together: two preparations recorded with one thermocouple, which the picrotoxin summaries enter as
two experiments; nothing in the file says so.
(4) Thermocouple dropouts (to -46.7 C) in three files, outside the analysis windows.
(5) LPAdvanceSamePeriod.mat includes 969_120, which has no spike file, and the code that made it is
not deposited. Spike times, temperature and the condition windows otherwise agree with the README,
and the raw intracellular trace and temperature of 969_113 share the spike times' clock.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")